# Sentinel â€” Clinical Fine-Tune for Mistral 7B
Uses QLoRA via Unsloth on a free Google Colab T4 GPU.
Output: GGUF file â†’ import into local Ollama as the `sentinel` model.

**How to run:** Runtime â†’ Run All (~30-45 min)

In [ ]:
# 1. Install Unsloth
import os, warnings
warnings.filterwarnings('ignore')
os.environ['HF_HUB_DISABLE_SYMLINKS_WARNING'] = '1'

!pip install unsloth -q
!pip install --upgrade --no-deps xformers trl peft accelerate bitsandbytes -q 2>/dev/null

In [ ]:
# 2. Load Mistral 7B Instruct in 4-bit
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/mistral-7b-instruct-v0.2-bnb-4bit",
    max_seq_length=2048,
    dtype=None,
    load_in_4bit=True,
)
print("Model loaded")

In [ ]:
# 3. Add LoRA adapters
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)
total = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable params: {total:,}")

In [ ]:
# 4. Upload training data (500 examples)
from google.colab import files
from datasets import Dataset
from unsloth.chat_templates import get_chat_template, standardize_sharegpt
import json

tokenizer = get_chat_template(tokenizer, chat_template='mistral')

print('Please upload sentinel_training_500.json')
uploaded = files.upload()
filename = list(uploaded.keys())[0]
training_data = json.loads(uploaded[filename])
print(f'Loaded {len(training_data)} examples')

dataset = Dataset.from_list(training_data)
dataset = standardize_sharegpt(dataset)

def format_chat(examples):
    outputs = tokenizer.apply_chat_template(
        examples['conversations'], tokenize=False
    )
    return {'text': outputs}

dataset = dataset.map(format_chat, batched=True)
print('Sample:\n', dataset[0]['text'][:300])


In [ ]:
# 5. Train! (~20-25 min for 500 examples)
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    max_seq_length=2048,
    dataset_num_proc=1,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        num_train_epochs=5,
        learning_rate=2e-4,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=1,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=42,
        output_dir="outputs",
        report_to="none",
    ),
)

trainer_stats = trainer.train()
print("\nTraining complete!")

In [ ]:
# 6. Save as GGUF for Ollama (~5 min)
model.save_pretrained_gguf(
    "sentinel_model",
    tokenizer,
    quantization_method="q4_k_m",
)
print("GGUF model saved to sentinel_model/")

In [ ]:
# 7. Download the GGUF file to your computer
from google.colab import files
import os

for f in os.listdir("sentinel_model"):
    if f.endswith(".gguf"):
        path = os.path.join("sentinel_model", f)
        gb = os.path.getsize(path) / (1024**3)
        print(f"Downloading {f} ({gb:.2f} GB)...")
        print("Save it somewhere you'll find (e.g. Downloads folder)")
        files.download(path)
        break

## Done!

After download:
1. Tell me where you saved the `.gguf` file
2. I'll create a Modelfile and import it into your local Ollama
3. Sentinel will automatically use the fine-tuned model